In [1]:
import pickle

with open('./output/store/pdf_files.pkl', 'rb') as f:
    pdf_texts,year = pickle.load(f)

In [2]:
import spacy
import numpy as np
import random
import re
seed = 42
random.seed(seed)
np.random.seed(seed)

In [3]:
nlp = spacy.load('en_core_web_lg')
nlp.max_length = 3000000

In [4]:
from spacy.lang.en.stop_words import STOP_WORDS
scientific_common_words = {
    "figure", "fig", "table", "et", "al", "etc", "eg", "i.e", "ie", 
    "eq", "conclusion", "introduction", "methods", "study", 
    "research", "author", "authors", "paper", "work", 
    "proposed", "presented", "based", "using", 
    "performed", "obtained", "observed", "used", "shown", "reported", 
    "significant", "important", "novel", "investigation", "algorithm", 
    "algorithms", "technique", "techniques", "parameters", 
    "parameter", "performance", "implemented", "implementation", "similar", 
    "different", "result", "respectively", "compare", "compared", 
    "comparison", "additional", "respectively", " ", "v.", "supra",
    "give", "follow", "know", "high","year", "term", "_",
    "find", "include", "tell" "try", "51ac", "pmpa", "id._note", "item", "coef",
    "interview", "percent", "go", "construct", "scale","000s", "ing", "pricewaterhousecooper",
    "analyse", "dmu?", "qsr?", "consequently", "furthermore", "atmo"
}
STOP_WORDS.update(scientific_common_words)
for word in scientific_common_words:
    print(f"Is '{word}' a stopword? {nlp.vocab[word].is_stop}")

Is 'furthermore' a stopword? True
Is 'reported' a stopword? True
Is 'figure' a stopword? True
Is '_' a stopword? False
Is '51ac' a stopword? True
Is 'techniques' a stopword? True
Is 'interview' a stopword? True
Is 'al' a stopword? True
Is 'fig' a stopword? True
Is 'proposed' a stopword? True
Is 'give' a stopword? True
Is 'additional' a stopword? True
Is 'respectively' a stopword? True
Is 'et' a stopword? True
Is 'authors' a stopword? True
Is 'important' a stopword? True
Is 'shown' a stopword? True
Is 'using' a stopword? True
Is 'eq' a stopword? True
Is 'v.' a stopword? False
Is 'study' a stopword? True
Is 'construct' a stopword? True
Is 'used' a stopword? True
Is 'novel' a stopword? True
Is '000s' a stopword? True
Is 'i.e' a stopword? False
Is 'algorithm' a stopword? True
Is 'implemented' a stopword? True
Is 'performed' a stopword? True
Is 'supra' a stopword? True
Is 'pricewaterhousecooper' a stopword? True
Is 'year' a stopword? True
Is 'comparison' a stopword? True
Is 'pmpa' a stopwor

In [5]:
from concurrent.futures import ThreadPoolExecutor, as_completed

with ThreadPoolExecutor() as executor:
    future_to_index = {executor.submit(nlp, text): index for index, text in enumerate(pdf_texts)}
    doc_objects = [None] * len(pdf_texts)

    for future in as_completed(future_to_index):
        index = future_to_index[future] 
        print(f"Completed: {index}")
        doc_objects[index] = future.result()

Completed: 2
Completed: 10
Completed: 9
Completed: 1
Completed: 8
Completed: 7
Completed: 4
Completed: 6
Completed: 5
Completed: 11
Completed: 3
Completed: 16
Completed: 0
Completed: 13
Completed: 12
Completed: 14
Completed: 17
Completed: 19
Completed: 20
Completed: 18
Completed: 15
Completed: 25
Completed: 29
Completed: 27
Completed: 23
Completed: 24
Completed: 30
Completed: 28
Completed: 26
Completed: 22
Completed: 21
Completed: 35
Completed: 32
Completed: 37
Completed: 34
Completed: 31
Completed: 43
Completed: 38
Completed: 33
Completed: 40
Completed: 46
Completed: 44
Completed: 36
Completed: 41
Completed: 48
Completed: 49
Completed: 47
Completed: 53
Completed: 39
Completed: 51
Completed: 42
Completed: 58
Completed: 50
Completed: 45
Completed: 56
Completed: 54
Completed: 52
Completed: 57
Completed: 62
Completed: 61
Completed: 60
Completed: 59
Completed: 65
Completed: 64
Completed: 67
Completed: 66
Completed: 71
Completed: 69
Completed: 55
Completed: 70
Completed: 68
Completed: 73
Co

In [23]:
pattern = r'\b(?![A-Za-z]+\b)\S+\b'
#pattern = r'\b[\w\d]+[^\w\d\s]+[\w\d]*\b|\b[\w\d]*[^\w\d\s]+[\w\d]+\b'
def filter_tokens(token):
    return (
        not token.is_stop                    
        and not token.is_punct               
        and not token.like_email       
        and not token.like_url        
        and not token.like_num          
        and not token.ent_type_ == "PERSON"
        and token.pos_ != "PROPN"       
        and len(token.text) > 2      
        and not re.match(pattern, token.text)
    )
def process_doc(index, doc):
    """
    Apply the filter_tokens function to a single Doc object.
    """
    return index, [token.lemma_.lower() for token in doc if filter_tokens(token) and token.lemma_.strip()]

filtered_tokens = [None] * len(doc_objects)
with ThreadPoolExecutor() as executor:
    futures = []
    for index, doc in enumerate(doc_objects):
        futures.append(executor.submit(process_doc, index, doc))

    for future in as_completed(futures):
        index, tokens = future.result() 
        print(f"Filtered: {index}")
        filtered_tokens[index] = tokens 
print(len(filtered_tokens))
print(len(filtered_tokens[0]))

Filtered: 7
Filtered: 6
Filtered: 19
Filtered: 2
Filtered: 11
Filtered: 21
Filtered: 10
Filtered: 18
Filtered: 9
Filtered: 17
Filtered: 13
Filtered: 8
Filtered: 14
Filtered: 4
Filtered: 12
Filtered: 1
Filtered: 16
Filtered: 5
Filtered: 0
Filtered: 15
Filtered: 3
Filtered: 20
Filtered: 23
Filtered: 24
Filtered: 25
Filtered: 27
Filtered: 26
Filtered: 29
Filtered: 28
Filtered: 30
Filtered: 22
Filtered: 34
Filtered: 35
Filtered: 37
Filtered: 36
Filtered: 38
Filtered: 40
Filtered: 39
Filtered: 43
Filtered: 41
Filtered: 44
Filtered: 46
Filtered: 45
Filtered: 47
Filtered: 48
Filtered: 49
Filtered: 51
Filtered: 53
Filtered: 52
Filtered: 54
Filtered: 56
Filtered: 55
Filtered: 58
Filtered: 57
Filtered: 62
Filtered: 60
Filtered: 64
Filtered: 63
Filtered: 65
Filtered: 67
Filtered: 68
Filtered: 70
Filtered: 69
Filtered: 71
Filtered: 61
Filtered: 73
Filtered: 50
Filtered: 74
Filtered: 72
Filtered: 76
Filtered: 77
Filtered: 80
Filtered: 78
Filtered: 42
Filtered: 81
Filtered: 84
Filtered: 59
Filtered:

In [24]:
unique_tokens = set()
for tokens in filtered_tokens:
    unique_tokens.update(tokens)
print(f"Number of unique tokens after 1st stage of processing: {len(unique_tokens)}")

Number of unique tokens after 1st stage of processing: 24849


In [25]:
with open('./output/store/filtered_tokens.pkl', 'wb') as f:
    pickle.dump((filtered_tokens,year), f)